# Breast Cancer Detection with Naive Bayes (Instructor Answer Key)

In this lab, you will build a **Breast Cancer Detection** classifier using **Gaussian Naive Bayes**.

**Goal:** Predict whether a tumor is **malignant** or **benign** using 30 numeric features derived from fine needle aspirate (FNA) images.

## Dataset citation
- UCI Machine Learning Repository (WDBC): Wolberg, W., Mangasarian, O., Street, N., & Street, W. (1993). *Breast Cancer Wisconsin (Diagnostic) [Dataset]*. UCI Machine Learning Repository. DOI: 10.24432/C5DW2B. citeturn0search0
- scikit-learn dataset loader documentation: `sklearn.datasets.load_breast_cancer`. citeturn0search1

## Implementation citations
- Gaussian Naive Bayes in scikit-learn: `sklearn.naive_bayes.GaussianNB`. citeturn1search0
- Train/test splitting utility: `sklearn.model_selection.train_test_split`. citeturn1search1
- Feature scaling: `sklearn.preprocessing.StandardScaler`. citeturn1search2

> Note: This is an educational example. In real medical settings, models must be validated clinically and used with appropriate regulatory and ethical safeguards.


## What you will do (step-by-step)

1. Load the Wisconsin Diagnostic Breast Cancer dataset from scikit-learn.
2. Convert it to a DataFrame and explore basic shape/class balance.
3. Split the data into train and test sets (with stratification).
4. Standardize numeric features using `StandardScaler` (fit on train, transform train/test).
5. Train a **Gaussian Naive Bayes** model.
6. Evaluate performance using accuracy, confusion matrix, and classification report.
7. Make a prediction for a single new sample.


In [1]:
# STEP 1: Import required libraries
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [2]:
# STEP 2: Load the dataset
data = load_breast_cancer()

X = data.data
y = data.target

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (569, 30)
y shape: (569,)


In [3]:
# STEP 3: Create a DataFrame for easier exploration
feature_names = data.feature_names
df = pd.DataFrame(X, columns=feature_names)

# Add the target column (0 = malignant, 1 = benign in this dataset)
df["target"] = y

df.head()


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [4]:
# STEP 4: Quick dataset checks
print(df["target"].value_counts())
print(df.describe().T.head(8))


target
1    357
0    212
Name: count, dtype: int64
                     count        mean         std        min        25%  \
mean radius          569.0   14.127292    3.524049    6.98100   11.70000   
mean texture         569.0   19.289649    4.301036    9.71000   16.17000   
mean perimeter       569.0   91.969033   24.298981   43.79000   75.17000   
mean area            569.0  654.889104  351.914129  143.50000  420.30000   
mean smoothness      569.0    0.096360    0.014064    0.05263    0.08637   
mean compactness     569.0    0.104341    0.052813    0.01938    0.06492   
mean concavity       569.0    0.088799    0.079720    0.00000    0.02956   
mean concave points  569.0    0.048919    0.038803    0.00000    0.02031   

                           50%       75%        max  
mean radius           13.37000   15.7800    28.1100  
mean texture          18.84000   21.8000    39.2800  
mean perimeter        86.24000  104.1000   188.5000  
mean area            551.10000  782.7000  2501.0

In [5]:
# STEP 5: Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Train size:", X_train.shape[0])
print("Test size :", X_test.shape[0])


Train size: 455
Test size : 114


In [6]:
# STEP 6: Standardize features
# IMPORTANT: Fit scaler on training data only, then transform both train and test
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

X_train_scaled[:3]


array([[-1.07200079e+00, -6.58424598e-01, -1.08808010e+00,
        -9.39273639e-01, -1.35939882e-01, -1.00871795e+00,
        -9.68358632e-01, -1.10203235e+00,  2.81062120e-01,
        -1.13231479e-01, -7.04860874e-01, -4.40938351e-01,
        -7.43948977e-01, -6.29804931e-01,  7.48061001e-04,
        -9.91572979e-01, -6.93759567e-01, -9.83284458e-01,
        -5.91579010e-01, -4.28972052e-01, -1.03409427e+00,
        -6.23497432e-01, -1.07077336e+00, -8.76534437e-01,
        -1.69982346e-01, -1.03883630e+00, -1.07899452e+00,
        -1.35052668e+00, -3.52658049e-01, -5.41380026e-01],
       [ 1.74874285e+00,  6.65017334e-02,  1.75115682e+00,
         1.74555856e+00,  1.27446827e+00,  8.42288215e-01,
         1.51985232e+00,  1.99466430e+00, -2.93045055e-01,
        -3.20179716e-01,  1.27567198e-01, -3.81382677e-01,
         9.40746962e-02,  3.17524379e-01,  6.39656015e-01,
         8.73892616e-02,  7.08450758e-01,  1.18215034e+00,
         4.26212305e-01,  7.47970186e-02,  1.22834212e+

In [7]:
# STEP 7: Train Gaussian Naive Bayes
model = GaussianNB()

model.fit(X_train_scaled, y_train)


,priors,None
,var_smoothing,1e-09


In [8]:
# STEP 8: Evaluate the model
y_pred = model.predict(X_test_scaled)

acc = accuracy_score(y_test, y_pred)
print("Accuracy:", round(acc, 4))

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n", cm)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=data.target_names))


Accuracy: 0.9298

Confusion Matrix:
 [[38  4]
 [ 4 68]]

Classification Report:

              precision    recall  f1-score   support

   malignant       0.90      0.90      0.90        42
      benign       0.94      0.94      0.94        72

    accuracy                           0.93       114
   macro avg       0.92      0.92      0.92       114
weighted avg       0.93      0.93      0.93       114



In [9]:
# STEP 9: Make a prediction for ONE sample
# We'll pick the first test row as an example.
sample = X_test_scaled[0].reshape(1, -1)

pred_class = model.predict(sample)[0]
pred_prob  = model.predict_proba(sample)[0]

print("Predicted class:", data.target_names[pred_class])
print("Probabilities [malignant, benign]:", np.round(pred_prob, 4))


Predicted class: malignant
Probabilities [malignant, benign]: [1. 0.]


## Exercises (short)

1. What does the Naive Bayes “independence assumption” mean in this medical dataset?
2. Try changing `test_size` from 0.20 to 0.30. Does accuracy change?
3. Without scaling, does GaussianNB performance change? Why might scaling help?
4. Which error is more dangerous here: predicting *benign* when it is *malignant*, or the reverse? Explain.
